In [ ]:
# -------------------------------
# Installing Modules
# -------------------------------
!pip install langchain langchain-community pandas pydantic langchain-mistralai

In [ ]:
# -------------------------------
# Importing Modules
# -------------------------------
import os
import csv
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.structured_output import ToolStrategy

In [ ]:
# -------------------------------
# 🔐 API Key
# -------------------------------
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

In [ ]:
# -------------------------------
# 📦 Schema
# -------------------------------
class Task(BaseModel):
    title: str = Field(description="Task name")
    priority: Literal["low", "medium", "high"]
    reason: str = Field(description="Why this priority was assigned")


class TaskList(BaseModel):
    tasks: List[Task]


In [ ]:
# -------------------------------
# 🛠️ Tool 1: Prioritize Tasks
# -------------------------------
@tool
def prioritize_tasks(tasks: List[str]) -> List[dict]:
    """Assign priority to tasks based on urgency keywords."""

    prioritized = []

    for task in tasks:
        task_lower = task.lower()

        if any(word in task_lower for word in ["urgent", "asap", "immediately"]):
            priority = "high"
            reason = "Contains urgency keywords"
        elif any(word in task_lower for word in ["today", "soon"]):
            priority = "medium"
            reason = "Time-bound but not urgent"
        else:
            priority = "low"
            reason = "No urgency detected"

        prioritized.append({
            "title": task,
            "priority": priority,
            "reason": reason
        })

    return prioritized

In [ ]:
# -------------------------------
# 🛠️ Tool 2: Save to CSV
# -------------------------------
@tool
def save_tasks_to_csv(tasks: List[dict], filename: str = "/content/tasks.csv") -> str:
    """
    Save prioritized tasks to a CSV file.
    """

    fieldnames = ["title", "priority", "reason"]

    with open(filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)

        high_priority = [task for task in tasks if task["priority"] == "high"]
        medium_priority = [task for task in tasks if task["priority"] == "medium"]
        low_priority = [task for task in tasks if task["priority"] == "low"]
        tasks = high_priority + medium_priority + low_priority

        writer.writeheader()
        for task in tasks:
            writer.writerow(task)

    return f"Tasks successfully saved to {filename}"

In [ ]:
# -------------------------------
# 🤖 LLM
# -------------------------------
llm = ChatMistralAI(
    model="mistral-small",
    temperature=0
)

In [ ]:
# -------------------------------
# 🧠 Agent
# -------------------------------
agent = create_agent(
    model=llm,
    tools=[prioritize_tasks, save_tasks_to_csv],
    response_format=ToolStrategy(schema=TaskList),
)

In [ ]:
# -------------------------------
# 🚀 Run Agent
# -------------------------------
user_input = """
Tomorrow I need to finish the report ASAP,
call my Manager,
Buy groceries
and prepare slides urgently for the youtube video.
"""

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                user_input +
                "\n\nAfter prioritizing, save the tasks into a CSV file."
            )
        }
    ]
})

print(response)